# Variant 5 — TaskTransformer (only for task) + UnifiedTransformer (Kendall loss)
Un modello predice le task, poi vengono passate a UnifiedTransformer che prevede regioni e task

In [ ]:
import os
import torch
import pandas as pd
from config import DATA_DIR

from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

from core.models.TaskTransformer import TaskTransformer
from core.models.UnifiedTransformer import UnifiedTransformer
from core.training import train_unified_onlyregion, train_task
from utils import get_decoding, get_encoding, hamming_distance, edit_distance_weighted_levenshtein

device = 'cuda' if torch.cuda.is_available() else 'cpu'
#print(device)
DATA_FILE = DATA_DIR / 'prepared_data.pt'

In [ ]:
from config import SEED
from utils import set_seed
set_seed(SEED)

In [ ]:
info = torch.load(DATA_FILE, map_location=device, weights_only=False)
n  = info['n']

data_task = {
    'train_tasks': info['data_tasks'][:n],
    'val_tasks': info['data_tasks'][n:],
}

data = {
    'train_tasks': info['data_tasks'][:n],
    'val_tasks': info['data_tasks'][n:],
    'train_regions': info['data_regions'][:n],
    'val_regions': info['data_regions'][n:],
    'train_times': info['data_times'][:n],
    'val_times': info['data_times'][n:],
}

vocab_size_tasks = info['vocab_size_tasks']
vocab_size_regions = info['vocab_size_regions']
num_regions = info['num_regions']
num_tasks = info['num_tasks']

decode_tasks = lambda b: [info['id_to_bit_tasks'][x]   for x in b]
decode_regions = lambda b: [info['id_to_bit_regions'][x] for x in b]
encode_tasks = lambda a: [info['bit_to_id_tasks'][tuple(x)]   for x in a]
encode_regions = lambda a: [info['bit_to_id_regions'][tuple(x)] for x in a]

net = info['net']

#print(f'vocab_tasks={vocab_size_tasks}, vocab_regions={vocab_size_regions}, n_train={n}')

In [ ]:
param_task_transformer_default = dict(block_size=256, n_embd=256, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)
param_unified_default = dict(block_size=64, n_embd=128, n_head=4, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)

params = torch.load(DATA_DIR / 'v5_best_params.pt', map_location=device, weights_only=False) if os.path.exists(DATA_DIR / 'v5_best_params.pt') else None

p_task = param_task_transformer_default
p_unified = param_unified_default
if params is not None:
    p_task = {**params['TaskTransformer'], 'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}
    p_unified = {**params['UnifiedTransformer'], 'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}

print('Task:', p_task)
print('Unified:', p_unified)

In [ ]:
model_task = TaskTransformer(
    task_vocab_size=vocab_size_tasks,
    block_size=p_task['block_size'],
    n_embd=p_task['n_embd'],
    dropout=p_task['dropout'],
    n_head=p_task['n_head'],
    n_layer=p_task['n_layer'],
).to(device)

best_val_task = train_task(model_task, data_task, p_task, device, data_key='tasks', printing=False)
print(f'TaskTransformer trained. Best val loss: {best_val_task:.4f}')

In [ ]:
model = UnifiedTransformer(
    vocab_size_region=vocab_size_regions,
    vocab_size_task=vocab_size_tasks,
    block_size=p_unified['block_size'],
    n_embd=p_unified['n_embd'],
    dropout=p_unified['dropout'],
    n_head=p_unified['n_head'],
    n_layer=p_unified['n_layer'],
    separated_task=True,
).to(device)

best_val_unified = train_unified_onlyregion(model, data, p_unified, device, printing=False)
print(f'UnifiedTransformer trained. Best val loss: {best_val_unified:.4f}')

#print(f'log_var_task={model.log_var_task.item():.4f}, log_var_region={model.log_var_region.item():.4f}, log_var_time={model.log_var_time.item():.4f}')

In [ ]:
max_new_tokens = 10000

sep_task_id = info['bit_to_id_tasks'][tuple([0]*num_tasks)]
sep_region_id = info['bit_to_id_regions'][tuple([0]*num_regions)]
sep_mask = (info['data_tasks'][:n] == sep_task_id) & (info['data_regions'][:n] == sep_region_id)
mean_sep_delta = info['data_times'][:n][sep_mask].float().mean().item()

context_task = torch.tensor(encode_tasks([[0]*num_tasks]), dtype=torch.long, device=device).unsqueeze(0)
context_region = torch.tensor(encode_regions([[0]*num_regions]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([mean_sep_delta], dtype=torch.float32, device=device).unsqueeze(0)

gen_task_ids, gen_region_ids, gen_times = [], [], []

for _ in range(max_new_tokens):
    next_task = model_task.predict_next_task(idx_task=context_task, block_size=p_task['block_size'])
    context_task = torch.cat((context_task, next_task), dim=1)
    context_task_aligned = context_task[:, 1:]   # allineamento offset

    _, next_region, next_time = model.predict_next(
        idx_region=context_region, idx_times=context_time,
        block_size=p_unified['block_size'], idx_task=context_task_aligned
    )
    context_region = torch.cat((context_region, next_region), dim=1)
    context_time = torch.cat((context_time, next_time), dim=1)

    gen_task_ids.append(next_task.item())
    gen_region_ids.append(next_region.item())
    gen_times.append(next_time.item())

decoded = []
for i in range(max_new_tokens):
    t_bits = [int(b) for b in decode_tasks([gen_task_ids[i]])[0]]
    r_bits = [int(b) for b in decode_regions([gen_region_ids[i]])[0]]
    combined = r_bits + t_bits
    decoded.append(combined)
    #print(f'{i:02d}: {combined} — time={round(gen_times[i],4)}')

In [ ]:
# Raggruppa la sequenza in tracce separate (separatore = vettore zero)
traces_generated, current = [], []
for step in decoded:
    current.append(step)
    if step == [0]*(num_regions+num_tasks):
        if len(current) > 1:
            traces_generated.append(current)
        current = []

#for i,trace in enumerate(traces_generated):
#    print(f"{i}: {trace}")'''

In [ ]:
MIN_CONTEXT_STEPS = 100

cumulative = 0
skip_n = 1
for trace in traces_generated:
    if cumulative >= MIN_CONTEXT_STEPS:
        break
    cumulative += len(trace)
    skip_n += 1

#print(f"Warm-up: {skip_n} tracce saltate ({cumulative} step) | valutazione su {len(traces_generated)-skip_n} tracce")

In [ ]:
traces_decoded = get_decoding(traces_generated, net.regions, net.tasks)
#print(traces_decoded)

In [ ]:
'''
PROBLEMA: Può generare tracce sfasate, con più eventi per step.
get_decoding non funziona sotto questo punto di vista. o meglio, funziona generando eventi in più (tipo 6 eventi da 4 step perchè ci sono 2 step generati male)
'''

classifier_dict_tasks = info['classifier_dict_tasks']
dict_task_step_encoding = info['dict_task_step_encoding']
current_trace_context = []
#traces_decoded_list = [step for trace in traces_decoded for step in trace]

tasks_previous = [0] * num_tasks
current_trace = 0
mae_values = []
unpredicted_time_steps = 0

for i, (step, t) in enumerate(zip(decoded, gen_times)):
    bits = [int(b) for b in step]
    is_sep = bits == [0]*(num_regions+num_tasks)
    tasks_step = bits[num_regions:]

    step_events = []
    for j, task in enumerate(tasks_step):
        if task != tasks_previous[j]:
            step_events.append(("start_" if task == 1 else "end_") + net.tasks[j])
    tasks_previous = tasks_step

    if len(step_events) == 1:
        event_name = step_events[0]
        if event_name in classifier_dict_tasks:
            rt, max_len = classifier_dict_tasks[event_name]
            ctx = list(reversed(current_trace_context))[:max_len]
            padded = ctx + ['PAD'] * (max_len - len(ctx))
            encoded = [dict_task_step_encoding.get(s, dict_task_step_encoding['PAD']) for s in padded]
            expected = round(rt.predict([encoded])[0], 4)
        else: # Non ci dovrebbe mai entrare in teoria
            expected = 0.0 if not current_trace_context else "n/d"
        if current_trace >= skip_n:
            if expected != "n/d":
                mae_values.append(abs(t - expected))
            else:
                unpredicted_time_steps += 1
        current_trace_context.append(event_name)
        note = ""
    elif len(step_events) == 0: # step che non genera eventi (es. cambia solo la regione)
        expected, note = "—", "(step senza evento)"
        unpredicted_time_steps += 1
    else: # step malformato: accende/spegne 2 task insieme
        expected, note = "—", f"(step ambiguo: {step_events})"
        unpredicted_time_steps += 1

    if is_sep:
        current_trace_context = []
        current_trace += 1

    #print(f'{i:02d}: {bits} — time={round(t,4)} | expected={expected} {note}')

mae = sum(mae_values) / len(mae_values) if mae_values else float('nan')
#print(f'\nTime MAE (ignorando tracce warm-up, {len(mae_values)} step): {mae:.4f}')

In [ ]:
# Allineamento con conformance checking pm4py
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])
silent_prefixes = start + end + loop

# Creo i parametri per l'allineamento
model_cost, sync_cost = {}, {}
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is None or (t.label is not None and t.label.startswith(silent_prefixes)): # Se è una transizione silente
        model_cost[t] = 0
        sync_cost[t] = 10000
    else: # Se è un task vero e proprio
        model_cost[t] = 10000
        sync_cost[t] = 0

alignment_params = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost,
}

# Creiamo l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)

aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=alignment_params)
#for i,trace in enumerate(aligned_traces):
#    print(f"{i}: {trace}")'''

In [ ]:
'''Codifichiamo le tracce allineate (per poi poterle confrontare con quelle generate dal transformer)'''

silent_prefixes = start + end + tuple(["back_L"])

aligned_traceEncoded_regions, aligned_traceEncoded_tasks = get_encoding(
    [[step for _, step in a['alignment'] if step and not step.startswith(silent_prefixes) and step != '>>']
     for a in aligned_traces],
    net.regions, net.tasks, net.open_clauses, net.end_clauses
)

#print(aligned_traceEncoded_regions)

df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)

In [ ]:
aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

costs = []
for gen, aln in zip(traces_generated[skip_n:], aligned_traces_encoded[skip_n:]):
    cost = edit_distance_weighted_levenshtein(gen, aln, num_regions+num_tasks, num_regions+num_tasks, hamming_distance)
    costs.append(cost)

n_eval = len(costs)
mean_cost = sum(costs) / n_eval if n_eval > 0 else float('nan')
var_cost = sum((c - mean_cost)**2 for c in costs) / n_eval if n_eval > 0 else 0.0
std_cost = var_cost ** 0.5
n_conformant = sum(1 for c in costs if c == 0)
pct_conformant = 100.0 * n_conformant / n_eval if n_eval > 0 else 0.0

#print(f'Tracce totali: {len(traces_generated)}  |  valutate (post warm-up): {n_eval}')
#print(f'Edit distance — media: {mean_cost:.4f}  std: {std_cost:.4f}')
#print(f'Tracce conformanti: {pct_conformant:.1f}%  ({n_conformant}/{n_eval})')

In [ ]:
import json
import fcntl
import numpy as np
from datetime import datetime
from config import RESULTS_DIR

VARIANT_NAME = "variant5_task_unified"

eval_generated = traces_generated[skip_n:]
eval_aligned   = aligned_traces_encoded[skip_n:]

# --- Metriche strutturali ---
print(f'\n=== RISULTATI VALUTAZIONE — {VARIANT_NAME} ===')
print(f'Tracce generate totali:   {len(traces_generated)}  (warm-up saltate: {skip_n})')
print(f'Tracce valutate:          {len(eval_generated)}')

print(f'\n-- Edit Distance (Levenshtein pesata) --')
print(f'Media:    {np.mean(costs):.4f}')
print(f'Mediana:  {np.median(costs):.4f}')
print(f'Std dev:  {np.std(costs):.4f}')
print(f'Min:      {np.min(costs):.4f}')
print(f'Max:      {np.max(costs):.4f}')
print(f'Tracce con edit_distance == 0:  {sum(1 for c in costs if c == 0)} / {len(costs)} ({100*sum(1 for c in costs if c == 0)/len(costs):.1f}%)')
print(f'Tracce con edit_distance <= 10:  {sum(1 for c in costs if c <= 10)} / {len(costs)} ({100*sum(1 for c in costs if c <= 10)/len(costs):.1f}%)')
print(f'Tracce con edit_distance <= 20:  {sum(1 for c in costs if c <= 20)} / {len(costs)} ({100*sum(1 for c in costs if c <= 20)/len(costs):.1f}%)')

# --- Lunghezze tracce ---
gen_lengths = [len(t) for t in eval_generated]
aln_lengths = [len(t) for t in eval_aligned]
print(f'\n-- Lunghezza tracce --')
print(f'Generate    — media: {np.mean(gen_lengths):.1f}, min: {min(gen_lengths)}, max: {max(gen_lengths)}')
print(f'Riferimento — media: {np.mean(aln_lengths):.1f}, min: {min(aln_lengths)}, max: {max(aln_lengths)}')

# --- Hamming step-by-step (solo tracce stessa lunghezza) ---
same_len_pairs = [(g, a) for g, a in zip(eval_generated, eval_aligned) if len(g) == len(a)]
if same_len_pairs:
    ham_costs = [sum(hamming_distance(gs, als) for gs, als in zip(g, a)) for g, a in same_len_pairs]
    print(f'\n-- Hamming diretta (solo {len(same_len_pairs)} tracce stessa lunghezza) --')
    print(f'Media: {np.mean(ham_costs):.4f}')
    print(f'Tracce perfette (Hamming==0): {sum(1 for h in ham_costs if h == 0)} / {len(same_len_pairs)}')
else:
    ham_costs = []

# --- Metriche temporali ---
print(f'\n-- Errore temporale (MAE delta t) --')
print(f'MAE medio:   {np.mean(mae_values):.6f}')
print(f'MAE mediano: {np.median(mae_values):.6f}')
print(f'Std dev:     {np.std(mae_values):.6f}')
print(f"Time steps valutati: {len(mae_values)}")
print(f"Time steps saltati per anomalie strutturali: {unpredicted_time_steps}")


# --- Raccolta risultati ---
results = {
    "variant":          VARIANT_NAME,
    "timestamp":        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "skip_n":           skip_n,
    "total_generated":  len(traces_generated),
    "edit_dist_mean":   float(np.mean(costs)),
    "edit_dist_median": float(np.median(costs)),
    "edit_dist_std":    float(np.std(costs)),
    "edit_dist_min":    float(np.min(costs)),
    "edit_dist_max":    float(np.max(costs)),
    "perfect_traces":   int(sum(1 for c in costs if c == 0)),
    "le10_traces":       int(sum(1 for c in costs if c <= 10)),
    "le20_traces":       int(sum(1 for c in costs if c <= 20)),
    "total_traces":     int(len(costs)),
    "perfect_pct":      float(100 * sum(1 for c in costs if c == 0) / len(costs)),
    "gen_len_mean":     float(np.mean(gen_lengths)),
    "aln_len_mean":     float(np.mean(aln_lengths)),
    "hamming_mean":     float(np.mean(ham_costs)) if ham_costs else None,
    "hamming_perfect":  int(sum(1 for h in ham_costs if h == 0)) if ham_costs else None,
    "time_skipped_steps": unpredicted_time_steps,
    "time_mae_mean":    float(np.mean(mae_values)),
    "time_mae_median":  float(np.median(mae_values)),
    "time_mae_std":     float(np.std(mae_values)),
    "training": {
        "task_best_val_loss": round(float(best_val_task), 4),
        "unified_best_val_loss": round(float(best_val_unified), 4),
    },
}

# Scrittura thread-safe nel JSON condiviso tra tutte le varianti
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
json_path = RESULTS_DIR / "all_variants_results.json"

with open(json_path, "a+", encoding="utf-8") as f:
    fcntl.flock(f, fcntl.LOCK_EX)
    f.seek(0)
    content = f.read().strip()
    data = json.loads(content) if content else []
    data.append(results)
    f.seek(0)
    f.truncate()
    json.dump(data, f, ensure_ascii=False, indent=2)
    fcntl.flock(f, fcntl.LOCK_UN)

#print(f'\nSalvato in {json_path}')